# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadzaheer3344/flyrankAi_ml_internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
# 1
import os, sys, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# Setup (Colab or local)
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 'declining' pages
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Dataset: {df.shape[0]} pages, {df.shape[1]} columns")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}")
print("\nFirst 3 rows:")
df.head(3)

Dataset: 30000 pages, 45 columns
Declining rate: 0.542

First 3 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1


In [8]:
#features for hand rule
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

# See top 10 pages flagged
top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
print("Top 10 pages flagged by hand rule:")
display_cols = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction", "hand_rule_score"]
top10[display_cols]

# Distribution of hand rule scores
print(f"\nPages with non-zero hand rule score: {(df['hand_rule_score'] > 0).sum()}")
print(f"Score range: {df['hand_rule_score'].min():.0f} to {df['hand_rule_score'].max():.0f}")

Top 10 pages flagged by hand rule:

Pages with non-zero hand rule score: 17
Score range: 0 to 61678


In [9]:
def precision_at_k(scores, labels, k):
    """Calculate Precision@K for a ranking"""
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values

print("HAND RULE PERFORMANCE:")
for k in [10, 20, 50, 100]:
    p = precision_at_k(df["hand_rule_score"], y, k)
    print(f"Precision@{k}: {p:.3f}  (~{round(p*k)} of top {k} are declining)")

print(f"\nDeclining rate in full dataset: {y.mean():.3f}")

HAND RULE PERFORMANCE:
Precision@10: 1.000  (~10 of top 10 are declining)
Precision@20: 0.900  (~18 of top 20 are declining)
Precision@50: 0.680  (~34 of top 50 are declining)
Precision@100: 0.630  (~63 of top 100 are declining)

Declining rate in full dataset: 0.542


In [10]:
# Define features (non-leaky, pre-decision signals)
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

# Train depth-2 tree
tree_depth2 = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_depth2.fit(X, y)

# Print the readable tree
print("DEPTH-2 DECISION TREE - READABLE RULE:")
print("="*60)
print(export_text(tree_depth2, feature_names=features))
print("="*60)

# Feature importance
importance_df = pd.DataFrame({
    'feature': features,
    'importance': tree_depth2.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFEATURE IMPORTANCE:")
print(importance_df.round(4))

DEPTH-2 DECISION TREE - READABLE RULE:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0


FEATURE IMPORTANCE:
                  feature  importance
2         impressions_90d      0.6690
0        content_age_days      0.2882
3            avg_position      0.0427
1  days_since_last_update      0.0000
4                     ctr      0.0000
5              word_count      0.0000


In [11]:
# tree predictions (probability of being declining)
tree_score = tree_depth2.predict_proba(X)[:, 1]

print("COMPARISON: HAND RULE vs DEPTH-2 TREE")
print("="*60)

for k in [10, 20, 50, 100]:
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    diff = tr - hr
    print(f"Precision@{k}:  Hand Rule {hr:.3f}   Tree {tr:.3f}   Difference: {diff:+.3f}")

# Check tied scores
unique_scores = len(np.unique(tree_score))
print(f"\nTree has only {unique_scores} unique scores (one per leaf)")
print(f"Pages with top score: {(tree_score == tree_score.max()).sum()}")

COMPARISON: HAND RULE vs DEPTH-2 TREE
Precision@10:  Hand Rule 1.000   Tree 0.600   Difference: -0.400
Precision@20:  Hand Rule 0.900   Tree 0.550   Difference: -0.350
Precision@50:  Hand Rule 0.680   Tree 0.600   Difference: -0.080
Precision@100:  Hand Rule 0.630   Tree 0.660   Difference: +0.030

Tree has only 4 unique scores (one per leaf)
Pages with top score: 17816


In [12]:
print("EXPERIMENT 1: VARYING TREE DEPTH")
print("="*60)

for depth in [2, 3, 4, 5]:
    tree = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    tree.fit(X, y)
    score = tree.predict_proba(X)[:, 1]

    p50 = precision_at_k(score, y, 50)
    unique_leaves = len(np.unique(score))

    print(f"Depth {depth}:")
    print(f"  Precision@50: {p50:.3f}")
    print(f"  Unique scores: {unique_leaves}")
    print(f"  Tree depth: {tree.get_depth()}")

    if depth <= 3:  # Only print small trees
        print(f"  Tree rules:")
        print(export_text(tree, feature_names=features)[:200] + "...\n")
    else:
        print(f"  Feature importance: {tree.feature_importances_.round(3)}\n")

EXPERIMENT 1: VARYING TREE DEPTH
Depth 2:
  Precision@50: 0.600
  Unique scores: 4
  Tree depth: 2
  Tree rules:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
| ...

Depth 3:
  Precision@50: 0.720
  Unique scores: 8
  Tree depth: 3
  Tree rules:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_po...

Depth 4:
  Precision@50: 0.680
  Unique scores: 16
  Tree depth: 4
  Feature importance: [0.221 0.003 0.585 0.101 0.062 0.029]

Depth 5:
  Precision@50: 0.800
  Unique scores: 28
  Tree depth: 5
  Feature importance: [0.227 0.01  0.545 0.107 0.085 0.026]



In [13]:
print("EXPERIMENT 2: SWAPPING FEATURES")
print("="*60)

# Different feature sets to test
feature_sets = {
    "Default": ["content_age_days", "days_since_last_update", "impressions_90d",
                "avg_position", "ctr", "word_count"],

    "Drop impressions": ["content_age_days", "days_since_last_update",
                         "avg_position", "ctr", "word_count"],

    "Add engagement": ["content_age_days", "days_since_last_update", "impressions_90d",
                       "avg_position", "ctr", "word_count", "engagement_rate"],

    "Minimal": ["content_age_days", "impressions_90d", "avg_position"]
}

results = []
for name, feats in feature_sets.items():
    # Check if features exist
    available_feats = [f for f in feats if f in df.columns]
    if len(available_feats) < len(feats):
        print(f"  Warning: Some features missing for {name}")

    X_test = df[available_feats].replace([np.inf, -np.inf], np.nan).fillna(0)

    tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
    tree.fit(X_test, y)

    score = tree.predict_proba(X_test)[:, 1]
    p50 = precision_at_k(score, y, 50)

    # First split feature
    first_split = available_feats[tree.tree_.feature[0]] if tree.tree_.feature[0] >= 0 else "None"
    results.append({
        'Feature Set': name,
        'Precision@50': p50,
        'First Split': first_split,
        'Features Used': len(available_feats)
    })

results_df = pd.DataFrame(results)
print("\nResults:")
print(results_df.round(3))

EXPERIMENT 2: SWAPPING FEATURES

Results:
        Feature Set  Precision@50      First Split  Features Used
0           Default           0.6  impressions_90d              6
1  Drop impressions           0.7     avg_position              5
2    Add engagement           0.6  impressions_90d              7
3           Minimal           0.6  impressions_90d              3


In [14]:
print("EXPERIMENT 3: TRAIN/TEST SPLIT VALIDATION")
print("="*60)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train set: {len(X_train)} pages")
print(f"Test set: {len(X_test)} pages")
print(f"Declining rate - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}\n")

# Train on train, evaluate on test
tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

# Scores on test set
train_score = tree.predict_proba(X_train)[:, 1]
test_score = tree.predict_proba(X_test)[:, 1]

print("PERFORMANCE ON TEST SET (in-sample vs out-of-sample):")
for k in [20, 50, 100]:
    # In-sample (train)
    p_train = precision_at_k(train_score, y_train, min(k, len(y_train)))
    # Out-of-sample (test)
    p_test = precision_at_k(test_score, y_test, min(k, len(y_test)))

    print(f"Precision@{k}:  Train {p_train:.3f}   Test {p_test:.3f}   Drop: {(p_train - p_test):+.3f}")

# Compare hand rule on test set
print("\nHAND RULE ON TEST SET:")
hr_scores_test = df.loc[X_test.index, "hand_rule_score"].values
hr_test = precision_at_k(hr_scores_test, y_test, 50)
print(f"Hand Rule Precision@50 (test): {hr_test:.3f}")
print(f"Tree Precision@50 (test): {precision_at_k(test_score, y_test, 50):.3f}")

EXPERIMENT 3: TRAIN/TEST SPLIT VALIDATION
Train set: 21000 pages
Test set: 9000 pages
Declining rate - Train: 0.542, Test: 0.542

PERFORMANCE ON TEST SET (in-sample vs out-of-sample):
Precision@20:  Train 0.600   Test 0.650   Drop: -0.050
Precision@50:  Train 0.700   Test 0.680   Drop: +0.020
Precision@100:  Train 0.610   Test 0.640   Drop: -0.030

HAND RULE ON TEST SET:
Hand Rule Precision@50 (test): 0.580
Tree Precision@50 (test): 0.680


In [15]:
print("LEAKAGE DEMONSTRATION - WHY YOU CAN'T FEED OUTCOME BACK IN")
print("="*60)

# Create leaky feature set
leaky_features = features + ["trend_pct"]
X_leaky = df[leaky_features].replace([np.inf, -np.inf], np.nan).fillna(0)

# Train leaky tree
leaky_tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
leaky_tree.fit(X_leaky, y)

# Evaluate
leaky_score = leaky_tree.predict_proba(X_leaky)[:, 1]
p50_leaky = precision_at_k(leaky_score, y, 50)

print(f"Leaky tree Precision@50: {p50_leaky:.3f}")
print(f"Non-leaky tree Precision@50: {precision_at_k(tree_score, y, 50):.3f}")
print(f"Improvement: {p50_leaky - precision_at_k(tree_score, y, 50):+.3f}")

print("\nLeaky tree rules (readable):")
print(export_text(leaky_tree, feature_names=leaky_features))

print("\n⚠️  PROBLEM: The tree just splits on 'trend_pct'")
print("Because 'trend_pct' is what 'trend_direction' is derived from!")
print("This creates perfect predictions but teaches nothing about actual drivers.")

LEAKAGE DEMONSTRATION - WHY YOU CAN'T FEED OUTCOME BACK IN
Leaky tree Precision@50: 1.000
Non-leaky tree Precision@50: 0.600
Improvement: +0.400

Leaky tree rules (readable):
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0


⚠️  PROBLEM: The tree just splits on 'trend_pct'
Because 'trend_pct' is what 'trend_direction' is derived from!
This creates perfect predictions but teaches nothing about actual drivers.


In [16]:
print("MY TURN - CUSTOM EXPERIMENT")
print("="*60)

# Try your own feature combinations or parameters
# Example: Test different class weights
print("Testing different class weights:")
for weight in [{0:1, 1:1}, {0:1, 1:2}, {0:1, 1:5}, {0:1, 1:10}]:
    tree = DecisionTreeClassifier(
        max_depth=3,
        class_weight=weight,
        random_state=42
    )
    tree.fit(X, y)
    score = tree.predict_proba(X)[:, 1]
    p50 = precision_at_k(score, y, 50)
    print(f"  Weight {weight}: Precision@50 = {p50:.3f}")

# Try adding custom feature
# Create a new feature: "position_quality" = 1/(avg_position + 1)
df["position_quality"] = 1 / (df["avg_position"] + 1)
custom_features = features + ["position_quality"]
X_custom = df[custom_features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree_custom = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_custom.fit(X_custom, y)
score_custom = tree_custom.predict_proba(X_custom)[:, 1]
p50_custom = precision_at_k(score_custom, y, 50)

print(f"\nWith custom 'position_quality' feature:")
print(f"  Precision@50: {p50_custom:.3f}")
print(f"  Tree rules:")
print(export_text(tree_custom, feature_names=custom_features))

MY TURN - CUSTOM EXPERIMENT
Testing different class weights:
  Weight {0: 1, 1: 1}: Precision@50 = 0.720
  Weight {0: 1, 1: 2}: Precision@50 = 0.720
  Weight {0: 1, 1: 5}: Precision@50 = 0.600
  Weight {0: 1, 1: 10}: Precision@50 = 0.700

With custom 'position_quality' feature:
  Precision@50: 0.600
  Tree rules:
|--- impressions_90d <= 5.50
|   |--- position_quality <= 0.57
|   |   |--- class: 0
|   |--- position_quality >  0.57
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



In [17]:
print("WEEK 2 - SUMMARY & KEY TAKEAWAYS")
print("="*60)

print("\n KEY FINDINGS:")
print("1. Hand-written rule works well at top of ranking")
print("   - Identifies clearly stale+visible pages")

print("\n2. Decision tree reveals the actual signal")
print("   - Feature importance shows what drives decline")
print("   - Readable rules make models interpretable")

print("\n3. Depth matters - but not everything")
print("   - Deeper trees can improve precision slightly")
print("   - But become harder to read/interpret")

print("\n4. FEATURE LEAKAGE IS DANGEROUS")
print("    Don't use features derived from the outcome")
print("    Don't use post-decision signals")
print("    Use only pre-decision observable signals")

print("\n5. Validation strategy matters")
print("   - In-sample performance overestimates")
print("   - Use train/test split or client-holdout")
print("   - Out-of-sample gap shows real performance")

print("\nYOUR RESEARCH QUESTION FOR WEEK 2:")
print("'What are the most important pre-decision signals")
print("that predict page decline, and how do they compare")
print("to simple heuristic rules?'")

print("\n BEST PRACTICES LEARNED:")
print(" Start with simple, readable models")
print(" Always validate out-of-sample")
print(" Check for leakage explicitly")
print(" Compare against intuitive baselines")

WEEK 2 - SUMMARY & KEY TAKEAWAYS

 KEY FINDINGS:
1. Hand-written rule works well at top of ranking
   - Identifies clearly stale+visible pages

2. Decision tree reveals the actual signal
   - Feature importance shows what drives decline
   - Readable rules make models interpretable

3. Depth matters - but not everything
   - Deeper trees can improve precision slightly
   - But become harder to read/interpret

4. FEATURE LEAKAGE IS DANGEROUS
    Don't use features derived from the outcome
    Don't use post-decision signals
    Use only pre-decision observable signals

5. Validation strategy matters
   - In-sample performance overestimates
   - Use train/test split or client-holdout
   - Out-of-sample gap shows real performance

YOUR RESEARCH QUESTION FOR WEEK 2:
'What are the most important pre-decision signals
that predict page decline, and how do they compare
to simple heuristic rules?'

 BEST PRACTICES LEARNED:
 Start with simple, readable models
 Always validate out-of-sample
 Chec

In [18]:
# Create summary table
summary = pd.DataFrame({
    'Model': ['Hand Rule', 'Depth-2 Tree', 'Depth-3 Tree', 'Depth-4 Tree', 'Leaky Tree'],
    'Precision@50': [
        precision_at_k(df['hand_rule_score'], y, 50),
        precision_at_k(tree_depth2.predict_proba(X)[:,1], y, 50),
        precision_at_k(DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42).fit(X,y).predict_proba(X)[:,1], y, 50),
        precision_at_k(DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42).fit(X,y).predict_proba(X)[:,1], y, 50),
        precision_at_k(leaky_tree.predict_proba(X_leaky)[:,1], y, 50)
    ]
})

print("MODEL COMPARISON SUMMARY:")
print(summary.round(3))

# Save to CSV
summary.to_csv("week2_model_comparison.csv", index=False)
print("\n Saved comparison to week2_model_comparison.csv")

MODEL COMPARISON SUMMARY:
          Model  Precision@50
0     Hand Rule          0.68
1  Depth-2 Tree          0.60
2  Depth-3 Tree          0.72
3  Depth-4 Tree          0.68
4    Leaky Tree          1.00

 Saved comparison to week2_model_comparison.csv


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.